# 13 - Kafka

Paste each Bash block into an Ubuntu terminal in order. Keep one terminal for this notebook. This uses the existing ZooKeeper-backed Kafka configuration. Complete the shared-image and monitoring notebooks first.

## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Services: zookeeper:2181 and kafka:9092 (internal clients); Kafka JVM metrics: kafka:9404; ZooKeeper native/JVM metrics: zookeeper:7000. PVCs: zookeeper-data (1 GiB, /data) and data-kafka-0 (5 GiB, /var/lib/kafka/data). Start ZooKeeper before Kafka. All examples run CLI clients inside a pod, so no external broker ports or laptop tunnel are needed. Topic data and offsets survive ordinary pod replacement.

## Apply ZooKeeper

```bash
kubectl apply -f kafka/zookeeper.yaml
```



## Wait for ZooKeeper

```bash
kubectl rollout status statefulset/zookeeper --timeout=600s
```



## Apply Kafka

```bash
kubectl apply -f monitoring/telemetry-config.yaml
kubectl apply -f kafka/kafka.yaml
```



## Wait for Kafka

```bash
kubectl rollout status statefulset/kafka --timeout=600s
```



## Create test-topic

```bash
kubectl exec kafka-0 -- kafka-topics.sh --bootstrap-server kafka:9092 --create --if-not-exists --topic test-topic --partitions 3 --replication-factor 1
```



## Describe test-topic

```bash
kubectl exec kafka-0 -- kafka-topics.sh --bootstrap-server kafka:9092 --describe --topic test-topic
```



## Produce a message

```bash
printf 'workspace kafka example\n' | kubectl exec -i kafka-0 -- kafka-console-producer.sh --bootstrap-server kafka:9092 --topic test-topic
```



## Consume the message

```bash
kubectl exec kafka-0 -- kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic test-topic --from-beginning --max-messages 1 --timeout-ms 15000 > .state/test-kafka-message.txt
cat .state/test-kafka-message.txt
grep -qx "workspace kafka example" .state/test-kafka-message.txt
```



## Inspect Kafka storage

```bash
kubectl get pvc zookeeper-data data-kafka-0
```



## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components zookeeper,kafka --jobs kafka-jmx,zookeeper-native
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Validate an independent client

```bash
bash validations/verify-data.sh
```

This also checks the existing MySQL sample database and Kafka delivery from a separate temporary client pod.

## Stop Kafka and ZooKeeper

```bash
bash stop-components.sh kafka
```

Broker and ZooKeeper PVCs remain.

## Resume Kafka and ZooKeeper

```bash
bash start-components.sh kafka
```



## Read the retained message

```bash
kubectl exec kafka-0 -- kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic test-topic --from-beginning --max-messages 1 --timeout-ms 15000 > .state/test-kafka-persisted.txt
grep -qx "workspace kafka example" .state/test-kafka-persisted.txt
```

